# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

# import ncem as nc
import scvi
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import squidpy as sq
import pandas as pd
sc.settings.set_figure_params(dpi=80)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
sc.settings.verbosity = 3           
sc.settings.set_figure_params(
    dpi=200,       
    dpi_save=400,  
    fontsize=14,    
    facecolor="white",
    vector_friendly=True,
)
sc.settings.figdir = 'out/'

# Load Data

Load post-cell2location deconvoluted data

In [ ]:

path = 'resulting_adata_st.h5ad'
adata = sc.read(path)

In [ ]:
# Set up cell abundance data
abundance_df = adata.obsm['q05_cell_abundance_w_sf'].copy()
abundance_df.columns = [col.replace("q05cell_abundance_w_sf_", "") for col in abundance_df.columns]
adata.obsm['q05_cell_abundance_w_sf'].columns = [col.replace("q05cell_abundance_w_sf_", "") for col in adata.obsm['q05_cell_abundance_w_sf'].columns]

# Broad cell type annotation (cancer, stroma, healthy, endothelial)

In [ ]:
def label_compartments_from_abundance(
    adata,
    abundance_df: pd.DataFrame,
    sample_col: str,
    *,
    tumor_types = ('Cancer cells',),
    fibroblast_types = ('MyoCAF','iCAF','Stromal cells'),
    endothelial_types = ('Endothelial cells',),
    healthy_types = ('Acinar cells','Islet cells'),   
    calibration_targets: dict | None = None, 
    # wherever we know from the original paper the expected tumor % etc from pathologist, mutations, etc. 
    out_prefix: str = "",      
    eps: float = 1e-8,
    store_colors: bool = True
):

    abundance_df = abundance_df.loc[adata.obs_names]
    def _uniq_filter(names):
        seen, out = set(), []
        for n in names:
            if (n in abundance_df.columns) and (n not in seen):
                out.append(n); seen.add(n)
        return out

    tumor_types       = _uniq_filter(tumor_types)
    fibroblast_types  = _uniq_filter(fibroblast_types)
    endothelial_types = _uniq_filter(endothelial_types)
    healthy_types     = _uniq_filter(healthy_types)

    # per-spot fractions 
    denom = abundance_df.sum(axis=1).replace(0, np.nan)
    frac = (abundance_df.T / denom).T.fillna(0.0)

    # scores
    tumor_score = frac[tumor_types].sum(axis=1)              if tumor_types else pd.Series(0.0, index=frac.index)
    caf_score   = frac[fibroblast_types].sum(axis=1)         if fibroblast_types else pd.Series(0.0, index=frac.index)
    endo_score  = frac[endothelial_types].sum(axis=1)        if endothelial_types else pd.Series(0.0, index=frac.index)

    if healthy_types:
        other_score = frac[healthy_types].sum(axis=1)
    else:
        other_score = (1.0 - (tumor_score + caf_score + endo_score)).clip(lower=0.0, upper=1.0)

    scores4 = pd.DataFrame({
        "Cancer":      tumor_score,
        "CAF":         caf_score,
        "Endothelial": endo_score,
        "Other":       other_score
    }, index=frac.index)

    stroma_total = scores4["CAF"] + scores4["Endothelial"]
    scores3 = pd.DataFrame({
        "Cancer": scores4["Cancer"],
        "Stroma": stroma_total,
        "Other":  scores4["Other"]
    }, index=frac.index)

    # basic labels
    # Tumor vs Stroma (stroma = CAF + Endothelial; 'Other' is excluded)
    tumor_vs_stroma_simple = np.where(scores4["Cancer"] > stroma_total, "Tumor", "Stroma")

    # Primary: Tumor if tumor dominates; else best of CAF/Endothelial/Other
    primary_simple = pd.Series(np.where(
        tumor_vs_stroma_simple == "Tumor",
        "Tumor",
        scores4[["CAF","Endothelial","Other"]].idxmax(axis=1)
    ), index=frac.index)

    # 4-way compartment label (cancer, endothelial, stroma, healthy/other)
    cancer_caf_endo_other = scores4.idxmax(axis=1)

    # for when we know how much tumor % from the original publication to correctly annotate tumor regions
    calib_report_rows = []
    tau_by_sample = {}
    labels_tvs = pd.Series(tumor_vs_stroma_simple, index=frac.index)
    labels_primary = pd.Series(primary_simple, index=frac.index)

    if calibration_targets and sample_col in adata.obs:
        sample_series = adata.obs[sample_col].astype(str)
        tumor_ratio = scores4["Cancer"] / (scores4["Cancer"] + stroma_total + eps)

        for sample_name, target_pct in calibration_targets.items():
            idx = sample_series[sample_series == str(sample_name)].index
            if len(idx) == 0:
                continue
            target = float(target_pct) / 100.0
            tr = tumor_ratio.loc[idx].astype(float)

            if tr.nunique() <= 1:
                tau = tr.median()
                is_tumor = tr >= tau
            else:
                q = 1.0 - target
                q = min(max(q, 0.0), 1.0)
                tau = tr.quantile(q, interpolation="linear")
                is_tumor = tr >= tau
                frac_t = is_tumor.mean()
                if frac_t > target and (tr > tau).mean() >= target - 1e-9:
                    is_tumor = tr > tau  # break ties if we overshoot

            tau_by_sample[sample_name] = float(tau)

            labels_tvs.loc[idx] = np.where(is_tumor, "Tumor", "Stroma")
            stromal_winner = scores4.loc[idx, ["CAF","Endothelial","Other"]].idxmax(axis=1)
            labels_primary.loc[idx] = np.where(is_tumor, "Tumor", stromal_winner)

            achieved = (labels_tvs.loc[idx] == "Tumor").mean() * 100.0
            calib_report_rows.append({
                "SampleID": sample_name,
                "TargetTumorPct": float(target_pct),
                "AchievedTumorPct": round(achieved, 2),
                "Tau_tumor_ratio": float(tau),
                "N_spots": int(len(idx))
            })

    # write to anndata
    p = f"{out_prefix}" if (out_prefix == "" or out_prefix.endswith("_")) else (out_prefix + "_")

    # scores
    adata.obsm[p + "compartment_scores4"] = scores4.values
    adata.obsm[p + "compartment_scores3"] = scores3.values
    adata.uns[p + "compartment_scores4_cols"] = list(scores4.columns)
    adata.uns[p + "compartment_scores3_cols"] = list(scores3.columns)

    # labels
    cats4 = ["Cancer","CAF","Endothelial","Other"]
    adata.obs[p + "Cancer_CAF_Endothelial_Other"] = pd.Categorical(cancer_caf_endo_other, categories=cats4, ordered=False)
    adata.obs[p + "Tumor_vs_Stroma"] = pd.Categorical(labels_tvs, categories=["Tumor","Stroma"], ordered=False)
    adata.obs[p + "PrimaryCompartment"] = pd.Categorical(labels_primary, categories=["Tumor","CAF","Endothelial","Other"], ordered=False)

    if tau_by_sample:
        adata.uns[p + "calibration_tau_by_sample"] = tau_by_sample

    if store_colors:
        # (Cancer, CAF, Endothelial, Other)
        adata.uns[p + "Cancer_CAF_Endothelial_Other_colors"] = ["#d62728", "#567FB3", "#E7B57E", "#808080"]
        # Tumor vs Stroma
        adata.uns[p + "Tumor_vs_Stroma_colors"] = ["#d62728", "#1f77b4"]
        # PrimaryCompartment (Tumor, CAF, Endothelial, Other)
        adata.uns[p + "PrimaryCompartment_colors"] = ["#d62728", "#567FB3", "#E7B57E", "#808080"]

    calib_report = pd.DataFrame(calib_report_rows).sort_values("SampleID") if calib_report_rows else pd.DataFrame()
    return calib_report


# Plot spatial grid

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

def plot_spatial_grid(
    adata,
    color: str,                    
    lib_key: str = "library_id",
    spatial_key: str = "spatial",
    img_res: str = "lowres",       
    with_img: bool = True,
    ncols: int = 8,
    spot_size_scale: float = 1.0, 
    rasterized: bool = True,
    figsize_per_panel=(2.2, 2.2),
    pdf_path: str = "spatial_panels.pdf",
):
    assert color in adata.obs.columns, f"{color} not in adata.obs"
    assert lib_key in adata.obs.columns, f"{lib_key} not in adata.obs"

    cats = (adata.obs[color].cat.categories
            if pd.api.types.is_categorical_dtype(adata.obs[color])
            else pd.Index(sorted(map(str, adata.obs[color].unique()))))
    if f"{color}_colors" in adata.uns:
        raw = list(adata.uns[f"{color}_colors"])
        palette = {c: raw[i % len(raw)] for i, c in enumerate(cats)}
    else:
        import matplotlib as mpl
        cmap = plt.get_cmap("tab20")
        palette = {c: cmap(i % 20) for i, c in enumerate(cats)}

    libs = list(pd.Index(adata.obs[lib_key].astype(str).unique()))
    n = len(libs)
    ncols = max(1, ncols)
    nrows = 1 if n <= ncols else ncols 
    per_page = ncols * ncols  

    def spot_size_points2(lib):
        scf = adata.uns[spatial_key][lib]["scalefactors"]
        pix_diam = scf.get("spot_diameter_fullres", 100.)
        scale = scf.get(f"tissue_{img_res}_scalef", scf.get("tissue_hires_scalef", 1.0))
        diam = pix_diam * float(scale) * spot_size_scale
        return (diam ** 2)

    with PdfPages(pdf_path) as pdf:
        for page_start in range(0, n, per_page):
            chunk = libs[page_start: page_start + per_page]
            rows = math.ceil(len(chunk) / ncols)
            fig, axes = plt.subplots(
                rows, ncols, figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * rows),
                squeeze=False
            )
            ax_iter = (ax for row in axes for ax in row)

            for i, lib in enumerate(chunk):
                ax = next(ax_iter)
                sel = (adata.obs[lib_key].astype(str) == lib).values
                if not np.any(sel):
                    ax.axis("off"); ax.set_title(lib, fontsize=9)
                    continue

                xy = adata.obsm["spatial"][sel, :].astype(float)

                if with_img and (spatial_key in adata.uns) and (lib in adata.uns[spatial_key]):
                    payload = adata.uns[spatial_key][lib]
                    if "images" in payload and img_res in payload["images"]:
                        img = payload["images"][img_res]
                        # scale coords
                        scf = payload.get("scalefactors", {})
                        scale = scf.get(f"tissue_{img_res}_scalef",
                                        scf.get("tissue_hires_scalef", 1.0))
                        xy_plot = xy * float(scale)
                        ax.imshow(img)
                    else:
                        xy_plot = xy
                else:
                    xy_plot = xy

                col = adata.obs[color].astype(str).values[sel]
                s_pts2 = spot_size_points2(lib)
                for c in cats:
                    m = (col == str(c))
                    if not np.any(m):
                        continue
                    ax.scatter(
                        xy_plot[m, 0], xy_plot[m, 1],
                        s=s_pts2, c=palette[c], marker="o", linewidths=0,
                        rasterized=rasterized, alpha=1.0
                    )

                ax.set_title(lib, fontsize=9)
                ax.set_xticks([]); ax.set_yticks([])
                ax.set_xlim(xy_plot[:,0].min()-20, xy_plot[:,0].max()+20)
                ax.set_ylim(xy_plot[:,1].max()+20, xy_plot[:,1].min()-20)  # flip y for image origin

            for ax in ax_iter:
                ax.axis("off")

            plt.tight_layout(w_pad=0.3, h_pad=0.3)
            pdf.savefig(fig, dpi=300, bbox_inches="tight")
            plt.close(fig)

    print(f"Saved to {pdf_path}")


In [ ]:
sq.pl.spatial_scatter(
    adata,
    color="Cancer_CAF_Endothelial_Other",
    library_key="library_id",
    img=False,
    title="",
    size=1.5,
    outline=False,
    legend_loc=None,
    frameon=False,
    ncols=8,
    na_color="#ffffff",
    save="cancer_stroma_healthy.pdf",  
)


# cell2location cell abundance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter

def plot_relabund_by_sample_q05(
    adata,
    *,
    abundance_obsm="q05_cell_abundance_w_sf",
    library_key="library_id",
    cancer_col="Cancer cells",
    combine_small_into_other=True,
    other_thresh=0.01,
    palette_dict=None,
    out_pdf=None,
    dpi=300,
    orientation="vertical",        
    cancer_color="#8B0000",        
    figsize_per_sample_w=0.1,    
    figsize_h=10,
):
    # abundance matrix
    if abundance_obsm not in adata.obsm:
        raise KeyError(f"'{abundance_obsm}' not found in adata.obsm")
    W = adata.obsm[abundance_obsm]
    if hasattr(W, "columns"):
        W = W.copy()
    else:
        cols = adata.uns.get("c2l_abundance_cols", None)
        if cols is None:
            raise KeyError(
                f"{abundance_obsm} is an array; please provide column names in "
                "adata.uns['c2l_abundance_cols']"
            )
        W = pd.DataFrame(W, index=adata.obs_names, columns=list(map(str, cols)))

    # sum per sample, then convert to fractions
    if library_key not in adata.obs:
        raise KeyError(f"'{library_key}' not found in adata.obs")
    samp = adata.obs[library_key].astype(str)
    W_sum = W.groupby(samp).sum()
    F = W_sum.div(W_sum.sum(axis=1), axis=0).fillna(0.0)

    #  collapse the cell types that have very few cells
    if combine_small_into_other:
        mean_frac = F.mean(axis=0)
        keep = mean_frac[mean_frac >= other_thresh].index.tolist()
        drop = [c for c in F.columns if c not in keep]
        if len(drop) > 0:
            F["Other"] = F[drop].sum(axis=1)
            F = F.drop(columns=drop)

    #  order cancer cells first first (then others by overall abundance)
    if cancer_col not in F.columns:
        raise KeyError(f"'{cancer_col}' not found among columns.")
    others = (F.drop(columns=[cancer_col]).mean(axis=0)
                .sort_values(ascending=False)).index.tolist()
    cols_order = [cancer_col] + others

    # sort samples by cancer fraction
    order_samples = F[cancer_col].sort_values(ascending=False).index
    F_plot = F.loc[order_samples, cols_order]

    #  palette
    if palette_dict is None:
        b = sns.color_palette("tab20b", 20)
        c = sns.color_palette("tab20c", 20)
        # interleave: b0, c0, b1, c1, ...
        base = [col for pair in zip(b, c) for col in pair]  # len=40

        if len(cols_order) > len(base):
            extra = sns.husl_palette(len(cols_order) - len(base), s=.65, l=.45)
            base = base + list(extra)

        palette_dict = {ct: base[i] for i, ct in enumerate(cols_order)}

    palette_dict[cancer_col] = cancer_color          
    palette_dict.setdefault("Other", (0.6, 0.6, 0.6))  

    # plotting
    n = F_plot.shape[0]
    if orientation == "vertical":
        fig_w = 5
        fig, ax = plt.subplots(figsize=(7, 10), dpi=dpi)
        bottoms = np.zeros(n)
        x = np.arange(n)
        for ct in cols_order:
            vals = F_plot[ct].values
            ax.bar(x, vals, bottom=bottoms, width=0.9, alpha=0.9,
                   color=palette_dict.get(ct, None), edgecolor="none", label=ct)
            bottoms += vals
        ax.set_ylim(0, 1)
        ax.set_xlim(-0.5, n - 0.5)
        ax.set_xlabel("Sample")
        ax.set_ylabel("Relative abundance (q05)")
        ax.grid(axis="y", color="#eeeeee", lw=0.8)
        ax.grid(False)                         # turn off grid
        sns.despine(ax=ax)                     # remove top/right spines
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))  # show 0–100%
        ax.tick_params(length=0)     
        #ax.set_title("Cell-type relative abundance per sample (Cancer first)")
    else:
        fig_h = 10
        fig, ax = plt.subplots(figsize=(10,10), dpi=dpi)
        left = np.zeros(n)
        y = np.arange(n)
        for ct in cols_order:
            vals = F_plot[ct].values
            ax.barh(y, vals, left=left, height=0.85,
                    color=palette_dict.get(ct, None), edgecolor="black", linewidth=0.3, label=ct)
            left += vals
        ax.set_xlim(0, 1)
        ax.set_yticks(y)
        ax.set_yticklabels(F_plot.index.tolist())
        ax.invert_yaxis()
        ax.set_xlabel("Relative abundance (q05)")
        ax.set_ylabel("Sample")
        ax.grid(axis="x", color="#eeeeee", lw=0.8)
        #ax.set_title("Cell-type relative abundance per sample (Cancer first)")

    ax.legend(title="Cell type", bbox_to_anchor=(1.02, 1), loc="best", frameon=False)
    plt.tight_layout()
    if out_pdf:
        fig.savefig(out_pdf, bbox_inches="tight")
    return F_plot, fig, ax


In [ ]:
F_plot, fig, ax = plot_relabund_by_sample_q05(
    adata,
    abundance_obsm="q05_cell_abundance_w_sf",
    library_key="library_id",
    cancer_col="Cancer cells",
    orientation="vertical",                 # flip to vertical
    cancer_color="#8B0000",                 # dark red
    out_pdf="relabund_q05_by_sample_vertical_cancer_first.pdf",
)
plt.show()


# Normalization for downstream gene-level analysis

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, batch_key="library_id")
sc.pp.pca(adata, use_highly_variable=True, n_comps=19)
sc.external.pp.bbknn(adata,batch_key="library_id")

# SERPINE1 / SERPINB2

### Core utilities

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy import stats
import matplotlib.pyplot as plt

GENE_COLOR = {"SERPINE1": "#8062A5", "SERPINB2": "#CF8D87"}
NEG_COLOR  = "#9e9e9e"

def _counts_vec(adata, gene, layer="counts"):
    if gene not in adata.var_names:
        raise KeyError(f"{gene} not found in adata.var_names")
    j = adata.var_names.get_loc(gene)
    X = adata.layers[layer] if (layer and layer in adata.layers) else adata.X
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    return X[:, j].astype(float)

def _abundance_df(adata, abundance_obsm):
    if abundance_obsm not in adata.obsm:
        raise KeyError(f"{abundance_obsm} not found in adata.obsm")
    W = adata.obsm[abundance_obsm]
    if isinstance(W, pd.DataFrame):
        return W.copy()
    cols = adata.uns.get(f"{abundance_obsm}_cols", None)
    if cols is None:
        cols = [f"ct_{i}" for i in range(np.asarray(W).shape[1])]
    return pd.DataFrame(np.asarray(W), index=adata.obs_names, columns=list(map(str, cols)))

import numpy as np

def um_per_px_fullres(adata, slide, spot_um=55.0):
    slide = str(slide)
    sf = adata.uns["spatial"][slide]["scalefactors"]
    d_full = sf.get("spot_diameter_fullres", np.nan)

    if not np.isfinite(d_full) or d_full <= 0:
        raise ValueError(f"Bad spot_diameter_fullres for slide {slide!r}: {d_full}")

    return float(spot_um) / float(d_full) 


### Define SEPRINE1/B2+ and - spots

In [ ]:
def gene_pos_neg_in_compartment(
    adata, gene,
    *, comp_key="Cancer_CAF_Endothelial_Other", compartment="Cancer",
    counts_layer="counts", threshold="percentile", percentile=75,
    per_slide=True, slide_key="library_id"
):
    comp   = adata.obs[comp_key].astype(str).values
    is_comp = (comp == str(compartment))
    slides = adata.obs[slide_key].astype(str).values
    g = _counts_vec(adata, gene, counts_layer)

    if threshold == "percentile":
        if per_slide:
            thr_map = {}
            for s in np.unique(slides[is_comp]):
                vals = g[(slides == s) & is_comp]
                vals = vals[np.isfinite(vals)]
                thr_map[s] = (np.percentile(vals, percentile) if vals.size else np.nan)
            thr = np.array([thr_map.get(s, np.nan) for s in slides], dtype=float)
        else:
            v = g[is_comp]
            v = v[np.isfinite(v)]
            thr_val = np.percentile(v, percentile) if v.size else np.nan
            thr = np.full(adata.n_obs, thr_val, dtype=float)
    else:
        thr = np.full(adata.n_obs, float(threshold), dtype=float)

    pos = is_comp & np.isfinite(g) & (g >= thr)
    neg = is_comp & np.isfinite(g) & (g <  thr)
    return pos, neg

from scipy import stats



### SERPINE1/B2+/- TME composition

In [ ]:
def tme_pos_vs_neg_composition(
    adata, pos_mask, neg_mask,
    *, abundance_obsm="q05_cell_abundance_w_sf",
    exclude=("Cancer cells",),
    test="mw",          
    fdr=True
):
    # get abundance table
    W = _abundance_df(adata, abundance_obsm)

    # exclude cancer 
    cancer_col = next((c for c in W.columns if c.lower().startswith("cancer")), None)

    # non-cancer columns
    excl = set(exclude or ())
    nonc_cols = [c for c in W.columns if c != cancer_col and c not in excl]

    # per-spot composition over non-cancer (rows sum to 1)
    Wnc = W[nonc_cols].astype(float)
    rowsum = Wnc.sum(axis=1).replace(0.0, np.nan)
    Wnc_prop = Wnc.div(rowsum, axis=0)

    # masks → index lists
    pos_idx = adata.obs_names[np.asarray(pos_mask, bool)]
    neg_idx = adata.obs_names[np.asarray(neg_mask, bool)]

    rows = []
    for ct in nonc_cols:
        a = Wnc_prop.loc[pos_idx, ct].to_numpy(dtype=float)
        b = Wnc_prop.loc[neg_idx, ct].to_numpy(dtype=float)

        # drop NaNs (can occur if a spot's non-cancer sum was 0)
        a = a[np.isfinite(a)]
        b = b[np.isfinite(b)]
        n_pos, n_neg = int(a.size), int(b.size)

        if n_pos == 0 and n_neg == 0:
            continue

        # means & diff
        m_pos = float(np.nanmean(a)) if n_pos else np.nan
        m_neg = float(np.nanmean(b)) if n_neg else np.nan
        diff  = m_pos - m_neg

        # Cohen's d (pooled SD)
        if n_pos > 1 and n_neg > 1:
            var_p = np.nanvar(a, ddof=1); var_n = np.nanvar(b, ddof=1)
            denom = np.sqrt(((n_pos-1)*var_p + (n_neg-1)*var_n) / (n_pos + n_neg - 2)) if (n_pos+n_neg-2) > 0 else np.nan
            d = (m_pos - m_neg) / denom if np.isfinite(denom) and denom > 0 else 0.0
        else:
            d = np.nan

        # p-value
        try:
            if test == "welch":
                pval = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit").pvalue if (n_pos and n_neg) else np.nan
            else:  # "mw"
                pval = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue if (n_pos and n_neg) else np.nan
        except Exception:
            pval = np.nan

        rows.append({
            "cell_type": ct,
            "pos_mean": m_pos,
            "neg_mean": m_neg,
            "diff": diff,
            "cohens_d": float(d) if np.isfinite(d) else np.nan,
            "pvalue": float(pval) if np.isfinite(pval) else np.nan,
            "n_pos": n_pos,
            "n_neg": n_neg,
        })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    # FDR correction
    if fdr and "pvalue" in out:
        try:
            from statsmodels.stats.multitest import multipletests
            out["padj"] = multipletests(out["pvalue"].fillna(1.0), method="fdr_bh")[1]
        except Exception:
            out["padj"] = out["pvalue"]

    # sort by |Cohen's d| (strongest standardized effects first)
    out = out.sort_values("cohens_d", key=lambda s: np.abs(s.fillna(0)), ascending=False)
    return out.reset_index(drop=True)




#### Plot TME composition shift

In [ ]:
def plot_tme_comp_shift(
    tme_stats,                      
    gene="SERPINE1",
    k=12,                          
    metric="diff",                 
    use_p="padj",                  
    annotate="stars",               
    neg_color="#9e9e9e",
    pos_color="#5A3CB8",
    figsize=(6, 4)
):
    df = tme_stats.copy()

    if "cell_type" in df.columns:
        df = df.set_index("cell_type")

    for col in ["pos_mean","neg_mean","diff","cohens_d","pvalue","padj"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if metric not in df.columns:
        raise KeyError(f"Column '{metric}' not found in tme_stats.")
    if "pos_mean" not in df.columns or "neg_mean" not in df.columns:
        raise KeyError("Expected 'pos_mean' and 'neg_mean' columns in tme_stats.")

    pos_side = df[df[metric] > 0].sort_values(metric, ascending=False).head(k)
    neg_side = df[df[metric] < 0].sort_values(metric, ascending=True).head(k)
    show = pd.concat([neg_side, pos_side], axis=0)
    show = show[~show.index.duplicated(keep="first")]   # avoid duplicate labels
    show = show.sort_values(metric)                     # negatives first, positives last

    labels = show.index.tolist()
    vals   = show[metric].to_numpy()
    y = np.arange(len(labels))

    colors = [neg_color if v <= 0 else pos_color for v in vals]

    fig, ax = plt.subplots(figsize=figsize)
    ax.barh(y, vals, color=colors, edgecolor="black", linewidth=0.6)
    ax.axvline(0, color="#444", lw=1)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)

    if annotate and (use_p in show.columns):
        pad = 0.02 * (1.1 * np.nanmax(np.abs(vals)) if np.isfinite(vals).any() else 1.0)
        for yi, ct in enumerate(labels):
            p = show.loc[ct, use_p]
            if annotate == "stars":
                tag = _p_to_stars(p)
            elif annotate == "value":
                tag = f"{use_p}={p:.2e}" if np.isfinite(p) else ""
            else:
                tag = ""
            if tag:
                x_end = vals[yi]
                ax.text(x_end + (pad if x_end >= 0 else -pad), yi, tag,
                        va="center", ha="left" if x_end >= 0 else "right",
                        fontsize=9, color="#333333")

    ax.set_xlabel(f"{metric} (positive → higher in {gene}+)",fontsize=9)
    ax.set_ylabel("Cell type",fontsize=9)
    ax.set_title(f"TME composition shift {gene}+ vs {gene}",fontsize=9)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    m = np.nanmax(np.abs(vals)) if np.isfinite(vals).any() else 1.0
    ax.set_xlim(-1.1*m, 1.1*m)

    plt.tight_layout()
    return fig, ax


#### SERPINE1/B2+/- TME differences by dataset - Figure 4P

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

def tme_pos_vs_neg_composition_by_dataset(
    adata, pos_mask, neg_mask,
    *,
    abundance_obsm="q05_cell_abundance_w_sf",
    exclude=("Cancer cells",),
    dataset_key="dataset",
    test="mw",            
    fdr=True,
    min_per_group=3        

    """
    Returns a  DataFrame with columns:
      ['dataset','cell_type','pos_mean','neg_mean','diff','cohens_d','pvalue','padj','n_pos','n_neg']
    The means are computed on within-noncancer composition
    FDR is applied *within each dataset* across its cell types.
    """
    if abundance_obsm not in adata.obsm:
        raise KeyError(f"{abundance_obsm} not found in adata.obsm")

    W = adata.obsm[abundance_obsm]
    if not isinstance(W, pd.DataFrame):
        W = pd.DataFrame(W, index=adata.obs_names)
    W.columns = list(map(str, W.columns))

    cancer_col = next((c for c in W.columns if c.lower().startswith("cancer")), None)
    ct_cols = [c for c in W.columns if c not in set(exclude or ())]
    nonc_cols = [c for c in ct_cols if c != cancer_col] if cancer_col in (W.columns) else ct_cols

    rowsum = W[nonc_cols].sum(axis=1).replace(0.0, np.nan)
    Wnc_prop = W[nonc_cols].div(rowsum, axis=0)  # each spot sums to 1 across non-cancer types

    if dataset_key in adata.obs:
        ds_vec = adata.obs[dataset_key].astype(str)
    else:
        ds_vec = pd.Series(["_all_"] * adata.n_obs, index=adata.obs_names)

    pos_idx = pd.Index(adata.obs_names[pos_mask])
    neg_idx = pd.Index(adata.obs_names[neg_mask])

    out_rows = []
    for ds in ds_vec.unique():
        ds_spots = ds_vec.index[ds_vec == ds]
        P = pos_idx.intersection(ds_spots)
        N = neg_idx.intersection(ds_spots)
        if len(P) < min_per_group or len(N) < min_per_group:
            # not enough data in this dataset → skip
            continue

        # per-cell-type stats within dataset
        pvals = []
        chunk = []
        for ct in nonc_cols:
            a = Wnc_prop.loc[P, ct].values
            b = Wnc_prop.loc[N, ct].values

            # means & Cohen’s d on composition scale
            mu_pos, mu_neg = np.nanmean(a), np.nanmean(b)
            var_a, var_b = np.nanvar(a, ddof=1), np.nanvar(b, ddof=1)
            na, nb = np.isfinite(a).sum(), np.isfinite(b).sum()
            pooled = np.sqrt(((max(na-1,1)*var_a + max(nb-1,1)*var_b) / max(na+nb-2,1))) if (na>1 and nb>1) else np.nan
            d = (mu_pos - mu_neg)/pooled if (np.isfinite(pooled) and pooled>0) else 0.0

            # test
            if test == "welch":
                _, pval = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
            else:
                try:
                    _, pval = stats.mannwhitneyu(a, b, alternative="two-sided")
                except Exception:
                    pval = np.nan

            pvals.append(pval)
            chunk.append({
                "dataset": ds,
                "cell_type": ct,
                "pos_mean": float(mu_pos),
                "neg_mean": float(mu_neg),
                "diff": float(mu_pos - mu_neg),
                "cohens_d": float(d),
                "pvalue": float(pval) if np.isfinite(pval) else np.nan,
                "n_pos": int(len(P)),
                "n_neg": int(len(N)),
            })

        # FDR within dataset
        if fdr and len(chunk):
            # treat NaNs safely
            pvals_arr = np.array([r["pvalue"] if np.isfinite(r["pvalue"]) else 1.0 for r in chunk], float)
            padj = multipletests(pvals_arr, method="fdr_bh")[1]
            for r, q in zip(chunk, padj):
                r["padj"] = float(q)
        else:
            for r in chunk:
                r["padj"] = np.nan

        out_rows.extend(chunk)

    return pd.DataFrame(out_rows)

# ----------  hashing for non-significant ----------
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def _hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def _rgb_to_hex(rgb):
    return "#" + "".join(f"{int(max(0,min(255,c))):02X}" for c in rgb)

def _gradient_hex(start_hex, end_hex, n):
    s = np.array(_hex_to_rgb(start_hex), float)
    e = np.array(_hex_to_rgb(end_hex), float)
    if n <= 1: 
        return [start_hex]
    cols = [ _rgb_to_hex(tuple((s + (e - s) * t).round())) for t in np.linspace(0, 1, n) ]
    return cols

def _sign_palettes_for_datasets(datasets, pos_shades=None, neg_shades=None):
    """
    Returns two lists aligned to datasets: pos_colors[j], neg_colors[j].
    Defaults are 3 dark-purple shades and 3 dark-grey shades; if you have >3 datasets,
    we generate a gradient.
    """
    n = len(datasets)
    # defaults = “three shades of dark purple / grey”
    default_pos = ["#3B1F7A", "#5A3CB8", "#7A5CE0"]  # dark → mid-dark purple
    default_neg = ["#4A4A4A", "#777777", "#9E9E9E"]  # dark → medium grey

    if pos_shades is None:
        if n <= 3:
            pos = default_pos[:n]
        else:
            # deep → mid purple gradient
            pos = _gradient_hex("#2E1766", "#7A5CE0", n)
    else:
        pos = list(pos_shades)
        if len(pos) < n:
            # extend with gradient between last two or toward a mid color
            pos += _gradient_hex(pos[-1], "#7A5CE0", n - len(pos) + 1)[1:]

    if neg_shades is None:
        if n <= 3:
            neg = default_neg[:n]
        else:
            # dark → lighter grey gradient
            neg = _gradient_hex("#3F3F3F", "#B0B0B0", n)
    else:
        neg = list(neg_shades)
        if len(neg) < n:
            neg += _gradient_hex(neg[-1], "#B0B0B0", n - len(neg) + 1)[1:]

    return pos, neg

### Plot per-dataset cell type abundance differences

In [ ]:
def plot_tme_comp_shift_grouped(
    tme_stats_by_ds: pd.DataFrame,
    *,
    gene="SERPINE1",
    datasets=None,         # ["Chen","Pei","Zhou"]
    k=12,                   # top diff cell types to plot
    order_by="mean_abs",   
    use_p="padj",           # which p column to use for hatching/annotation
    p_cut=0.05,             # hatch if p >= p_cut
    annotate=None,          
    mode="grouped",        
    pos_shades=None,       
    neg_shades=None,        
    bar_gap=0.15,           
    bar_alpha=0.95,
    figsize=None
):
    """
    Single axes; identical cell-type order across datasets.
    Positive bars use dataset-specific purple shades; negative bars use grey shades.
    Non-significant bars are hatched.
    """
    if tme_stats_by_ds is None or tme_stats_by_ds.empty:
        raise ValueError("Empty tme_stats_by_ds")

    req = {"dataset","cell_type","diff"}
    miss = req - set(tme_stats_by_ds.columns)
    if miss:
        raise KeyError(f"Missing required columns: {sorted(miss)}")

    df = tme_stats_by_ds.copy()
    pcol = use_p if use_p in df.columns else ("padj" if "padj" in df.columns else "pvalue")
    if pcol not in df.columns:
        df[pcol] = np.nan

    if datasets is None:
        datasets = list(pd.unique(df["dataset"]))
    df = df[df["dataset"].isin(datasets)].copy()
    if df.empty:
        raise ValueError("No rows for the requested datasets")

    pivot = df.pivot_table(index="cell_type", columns="dataset", values="diff", aggfunc="mean").reindex(columns=datasets)
    if order_by.startswith("ref:"):
        ref = order_by.split(":",1)[1]
        if ref not in pivot.columns: 
            raise ValueError(f"order_by='{order_by}' but '{ref}' not in datasets")
        agg = pivot[ref].abs().fillna(0)
    else:
        agg = pivot.abs().mean(axis=1).fillna(0)
    top_cts = agg.sort_values(ascending=False).head(k).index.tolist()

    order = pivot.loc[top_cts].mean(axis=1).sort_values().index.tolist()

    D = df.pivot_table(index="cell_type", columns="dataset", values="diff",   aggfunc="mean").reindex(index=order, columns=datasets)
    P = df.pivot_table(index="cell_type", columns="dataset", values=pcol,     aggfunc="min" ).reindex(index=order, columns=datasets)

    n_ct, n_ds = len(order), len(datasets)
    if figsize is None:
        w = max(6.0, 1.4 + n_ds * 1.2)
        h = max(2.8, 0.35 * n_ct + 1.2)
        figsize = (w, h)

    pos_colors, neg_colors = _sign_palettes_for_datasets(datasets, pos_shades, neg_shades)

    fig, ax = plt.subplots(figsize=figsize)
    y = np.arange(n_ct)
    m = np.nanmax(np.abs(D.values)) if D.size else 1.0
    m = 1.0 if not np.isfinite(m) or m == 0 else m
    ax.set_xlim(-1.1*m, 1.1*m)
    ax.axvline(0, color="#444", lw=1)

    def _stars(p):
        if not np.isfinite(p): return ""
        return "***" if p < 1e-3 else ("**" if p < 1e-2 else ("*" if p < 5e-2 else ""))

    if mode == "grouped":
        total_w = min(0.8, 0.65 + 0.05 * n_ds)
        step = total_w / max(n_ds, 1)
        left = -0.5 * total_w + step/2.0

        for j, ds in enumerate(datasets):
            xvals = D[ds].values
            yy = y + (left + j*step)
            bar_cols = [ (pos_colors[j] if (np.isfinite(v) and v > 0) else neg_colors[j]) for v in xvals ]
            bars = ax.barh(yy, xvals, height=step * (1.0 - bar_gap), color=bar_cols,
                           alpha=bar_alpha, edgecolor="black", linewidth=0.6, label=str(ds))
            pvals = P[ds].values
            for b, p in zip(bars, pvals):
                if not (np.isfinite(p) and p < p_cut):
                    b.set_hatch("///"); b.set_alpha(0.85)
            if annotate in ("stars","text"):
                xpad = 0.02 * m
                for yi, (v, p) in enumerate(zip(xvals, pvals)):
                    if not np.isfinite(v): continue
                    tag = _stars(p) if annotate == "stars" else (f"p={p:.2e}" if np.isfinite(p) else "")
                    if tag:
                        x = v + (xpad if v >= 0 else -xpad)
                        ha = "left" if v >= 0 else "right"
                        ax.text(x, yy[yi], tag, va="center", ha=ha, fontsize=8, color="#333")
    else:  
        heights = np.linspace(0.82, 0.42, n_ds)
        for j, ds in enumerate(datasets):
            xvals = D[ds].values
            bar_cols = [ (pos_colors[j] if (np.isfinite(v) and v > 0) else neg_colors[j]) for v in xvals ]
            bars = ax.barh(y, xvals, height=heights[j], color=bar_cols,
                           alpha=bar_alpha, edgecolor="black", linewidth=0.6, label=str(ds))
            pvals = P[ds].values
            for b, p in zip(bars, pvals):
                if not (np.isfinite(p) and p < p_cut):
                    b.set_hatch("///"); b.set_alpha(0.85)
            if annotate in ("stars","text"):
                xpad = 0.02 * m
                for yi, (v, p) in enumerate(zip(xvals, pvals)):
                    if not np.isfinite(v): continue
                    tag = _stars(p) if annotate == "stars" else (f"p={p:.2e}" if np.isfinite(p) else "")
                    if tag:
                        x = v + (xpad if v >= 0 else -xpad)
                        ha = "left" if v >= 0 else "right"
                        ax.text(x, yi, tag, va="center", ha=ha, fontsize=8, color="#333")

    ax.set_yticks(y); ax.set_yticklabels(order, fontsize=8)
    ax.set_xlabel(f"Higher in {gene}- ←→ Higher in {gene}+", fontsize=7)
    ax.set_title(f"TME composition shift", fontsize=8)
    ax.tick_params(axis='x', labelsize=8) 
    ax.legend(frameon=False, ncol=1, loc="right", fontsize=8)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.tight_layout()
    return fig, ax

#### SERPINE1/B2+  TME per compartment (cancer/caf/endothelial) - Figure 4Q

In [ ]:
def tme_in_gene_pos_by_compartment(
    adata, gene="SERPINE1",
    *, comp_key="Cancer_CAF_Endothelial_Other",
    compartments=("Cancer","CAF","Endothelial"),
    abundance_obsm="q05_cell_abundance_w_sf",
    counts_layer="counts",
    exclude=("Cancer cells",),
    normalize_within_noncancer=True,
    posneg_obs_col=None,         
    pos_tag="+"
):

    import numpy as np, pandas as pd

    # matrices
    comp = adata.obs[comp_key].astype(str).values
    W = _abundance_df(adata, abundance_obsm)

    cancer_col = next((c for c in W.columns if c.lower().startswith("cancer")), None)
    ct_cols = [c for c in W.columns if c not in set(exclude or ())]
    if normalize_within_noncancer and (cancer_col is not None):
        nonc = [c for c in ct_cols if c != cancer_col]
        rowsum = W[nonc].sum(axis=1).replace(0.0, np.nan)
        W_use = W[nonc].div(rowsum, axis=0)
    else:
        W_use = W[ct_cols].copy()

    if posneg_obs_col is not None and posneg_obs_col in adata.obs:
        labels = adata.obs[posneg_obs_col].astype(str).values
        def is_pos_in_compartment(cpt):
            return labels == f"{cpt}{pos_tag}"
    else:
        g = _counts_vec(adata, gene, counts_layer)
        gfinite = np.isfinite(g)
        def is_pos_in_compartment(cpt):
            return (comp == cpt) & gfinite & (g >= 1.0)

    means = {}
    for cpt in compartments:
        m = is_pos_in_compartment(cpt)
        means[cpt] = (W_use.loc[adata.obs_names[m]].mean(axis=0)
                      if np.count_nonzero(m) else pd.Series(index=W_use.columns, dtype=float))
    return pd.DataFrame(means)


def tme_in_gene_pos_by_compartment_pvals(
    adata, gene="SERPINE1",
    *, comp_key="Cancer_CAF_Endothelial_Other",
    compartments=("Cancer","CAF","Endothelial"),
    abundance_obsm="q05_cell_abundance_w_sf",
    counts_layer="counts",
    exclude=("Cancer cells",),
    normalize_within_noncancer=True,
    test="mw",   
    fdr=True,
    
    posneg_obs_col=None,
    pos_tag="+"
):

    import numpy as np, pandas as pd
    from scipy import stats

    comp = adata.obs[comp_key].astype(str).values
    W = _abundance_df(adata, abundance_obsm)

    cancer_col = next((c for c in W.columns if c.lower().startswith("cancer")), None)
    ct_cols = [c for c in W.columns if c not in set(exclude or ())]
    if normalize_within_noncancer and (cancer_col is not None):
        nonc = [c for c in ct_cols if c != cancer_col]
        rowsum = W[nonc].sum(axis=1).replace(0.0, np.nan)
        W_use = W[nonc].div(rowsum, axis=0)
        cell_types = nonc
    else:
        W_use = W[ct_cols].copy()
        cell_types = ct_cols

    if posneg_obs_col is not None and posneg_obs_col in adata.obs:
        labels = adata.obs[posneg_obs_col].astype(str).values
        gene_pos_all = np.array([s.endswith(pos_tag) for s in labels], dtype=bool)
    else:
        g = _counts_vec(adata, gene, counts_layer)
        gene_pos_all = np.isfinite(g) & (g >= 1.0)

    P = pd.DataFrame(index=cell_types, columns=list(compartments), dtype=float)
    for cpt in compartments:
        this = (comp == cpt) & gene_pos_all
        other = (comp != cpt) & gene_pos_all
        if this.sum() == 0 or other.sum() == 0:
            P.loc[:, cpt] = np.nan
            continue

        i_this = adata.obs_names[this]
        i_other = adata.obs_names[other]
        for ct in cell_types:
            a = W_use.loc[i_this, ct].astype(float).values
            b = W_use.loc[i_other, ct].astype(float).values
            if not (np.isfinite(a).any() and np.isfinite(b).any()):
                P.loc[ct, cpt] = np.nan; continue
            try:
                if test == "ttest":
                    p = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit").pvalue
                else:
                    p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
            except Exception:
                p = np.nan
            P.loc[ct, cpt] = float(p)

    if fdr:
        from statsmodels.stats.multitest import multipletests
        flat = P.values.flatten()
        ok = np.isfinite(flat)
        q = np.full_like(flat, np.nan, dtype=float)
        if ok.sum() > 0:
            q[ok] = multipletests(flat[ok], method="fdr_bh")[1]
        Q = pd.DataFrame(q.reshape(P.shape), index=P.index, columns=P.columns)
    else:
        Q = P.copy()

    return P, Q



In [ ]:
def plot_compartment_heatmap(
    M, gene="SERPINE1", cmap="YlGnBu",
    ytick_fontsize=7, xtick_fontsize=8, title_fontsize=12,
    padj=None,                
    fdr_cut=0.05,
    hatch="///",
    hatch_color="black",
    hatch_alpha=0.65,
    linewidths=0.4, linecolor="white",
):

    import seaborn as sns
    from matplotlib.ticker import FixedLocator, FixedFormatter
    from matplotlib.patches import Rectangle

    H = M.copy()
    fig_w = max(3.5, 0.32 * H.shape[1] + 1.2)
    fig_h = max(2.6, 0.26 * H.shape[0] + 1.0)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    # base heatmap
    sns.heatmap(H, cmap=cmap, linewidths=linewidths, linecolor=linecolor, cbar=True, ax=ax)

    ax.xaxis.set_major_locator(FixedLocator(np.arange(H.shape[1]) + 0.5))
    ax.xaxis.set_major_formatter(FixedFormatter(list(H.columns)))
    ax.yaxis.set_major_locator(FixedLocator(np.arange(H.shape[0]) + 0.5))
    ax.yaxis.set_major_formatter(FixedFormatter(list(H.index)))

    ax.tick_params(axis="x", labelsize=xtick_fontsize, rotation=90, length=0)
    ax.tick_params(axis="y", labelsize=ytick_fontsize, rotation=0, length=0)
    ax.set_title(f"TME in {gene}+ spots", fontsize=title_fontsize, pad=4)

    if padj is not None:
        Q = pd.DataFrame(padj, copy=True)
        Q = Q.reindex(index=H.index, columns=H.columns)
        for i, ct in enumerate(H.index):
            for j, cpt in enumerate(H.columns):
                q = Q.loc[ct, cpt]
                if not np.isfinite(q) or (q >= float(fdr_cut)):
                    rect = Rectangle(
                        (j, i), 1, 1,
                        facecolor=(1,1,1,0),   
                        edgecolor=(0,0,0,0),  
                        hatch=hatch,
                        linewidth=0.0
                    )
                    ax.add_patch(rect)
                    rect.set_edgecolor(hatch_color)
                    rect.set_alpha(hatch_alpha)
                    rect.set_zorder(10)     

    try: sns.despine(ax=ax, left=False, bottom=False)
    except: pass
    plt.tight_layout()
    return fig, ax
